In [8]:
import requests

base_url = "http://192.168.157.163:8009"

### 下載模型到本地暫存區

In [9]:
# model_name = "google/gemma-3-270m-it"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model_name = "Qwen/Qwen3-1.7B"

data = {
    "model_source": "huggingface",
    "model_name": model_name
}

url = f"{base_url}/models/download"
response = requests.post(url, json=data)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print(result["message"])
    print("模型暫存區:", result["details"])
    model_path = result["details"]
else:
    print(f"錯誤: {response.status_code} - {response.text}")
    


模型下載請求已處理
模型暫存區: /app/tmp/models/Qwen_Qwen3-1.7B


### 下載資料集到本地暫存區

In [ ]:
# dataset_name = "rajpurkar/squad_v2"
dataset_name = "yentinglin/TaiwanChat"
# dataset_name = "kigner/ruozhiba-llama3-tt"
# dataset_name = "HuggingFaceH4/ultrachat_200k"
# dataset_name = "unsloth/OpenMathReasoning-mini"
# dataset_name = "Mxode/Chinese-Reasoning-Distil-Data"

data = {
  "dataset_name": dataset_name,
  "dataset_source": "huggingface",
  "extract_multimedia": False
}

# url = f"{base_url}/datasets/download_from_network"
# response = requests.post(url, json=data)

# # 檢查回應狀態
# if response.status_code == 200:
#     result = response.json()
#     print(result["message"])
#     print("資料集暫存區:", result["local_path"])
#     dataset_path = result["local_path"]
# else:
#     print(f"錯誤: {response.status_code} - {response.text}")


# 已下載過資料集 為節省時間直接指定path
dataset_path = "/app/tmp/datasets/yentinglin_TaiwanChat"
# dataset_path = "/app/tmp/datasets/kigner_ruozhiba-llama3-tt"
# dataset_path = "/app/tmp/datasets/HuggingFaceH4_ultrachat_200k"
# dataset_path = "/app/tmp/datasets/unsloth_OpenMathReasoning-mini"
# dataset_path = "/app/tmp/datasets/Mxode_Chinese-Reasoning-Distil-Data"

print(dataset_name, dataset_path)

yentinglin/TaiwanChat /app/tmp/datasets/yentinglin_TaiwanChat


### 提交訓練任務

In [ ]:
dataset_config = {
	"dataset_name_or_path": dataset_name,
	"cache_dir": dataset_path,
	"max_length": 2048,
	"train_size": 200, 						# 訓練資料筆數
	"val_size": 50,							# 驗證資料筆數
	"train_split_name": "train",            # 指定split
	"num_workers": 8, 						# 用多少個 CPU 子程序來並行載入資料
	"column_mapping": {
		# "context": "context",				# 用於instruction，如SQuAD 或 RAG 任務的上下文/文章欄位。
		"messages": "messages",
        "reasoning": "reasoning",           # 用於instruction，如模型思考過程。
    	"input": "prompt",					# 用於instruction，如提問或指令。
    	"output": "response",				# 用於instruction，如模型期望的回答。
        # "text": "text"					# 用於text generation
	}
}

experiment_name = "_".join(
    [
        model_name.split('/')[1].split('-')[0],
        dataset_name.split('/')[1], 
        "finetune"
    ]
)

training_config = {
    "experiment_name": experiment_name,
    "model_name_or_path": model_path,
    "max_epochs": 10,
	"batch_size": 1,
	"gradient_accumulation_steps": 4,
	"learning_rate": 0.00002,
	"logging_steps": 5,
	"lora_config": {
		"bias": "none",
		"lora_alpha": 16,
		"lora_dropout": 0.05,
		"r": 8,
		"target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"]
	},
	"quantization_config": {
		"load_in_4bit": True,
		"bnb_4bit_quant_type": "nf4",        # 通常 nf4 比 fp4 效果好
		"bnb_4bit_use_double_quant": True,   # 雙重量化，進一步節省記憶體
		"bnb_4bit_compute_dtype": "bfloat16" # 計算時使用的精度 (建議與 use_bfloat16 一致)
    },
	"use_bfloat16": True,
	"use_flash_attn": False,
	"val_check_interval": 1.0,
	"warmup_steps": 100,
	"weight_decay": 0.01
}

submission = {
    "training_config": training_config,
    "dataset_config": dataset_config,
    "task_type": "instruction", # 對於文字生成任務，改成text
    "select_multiple_gpus": False,
    "vram_budget_gb": 15 # 估算訓練時所需vram
}

url = f"{base_url}/training/submit"
response = requests.post(url, json=submission)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print("任務 ID:", result["job_id"])
    print("狀態:", result["status"])
    print("使用的GPU:", result["gpu_id"])
    job_id = result["job_id"]
else:
    print(f"錯誤: {response.status_code} - {response.text}")

任務 ID: 0871ef8f2d2b4a
狀態: running
使用的GPU: [1]


### 追蹤任務進度

In [12]:
url = f"{base_url}/training/status/{job_id}"
response = requests.get(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    print("狀態", result["status"])
    display(result["metrics"])
else:
    print(f"錯誤: {response.status_code} - {response.text}")


狀態 running


{'epoch': 0.0,
 'estimated_time_remaining_seconds': 529,
 'progress_percentage': 7.0,
 'train_loss_step': 1.7578715085983276,
 'lr-AdamW': 7.800000000000002e-06,
 'mlflow_run_id': 'b269e25e9f13419a979eeb31eb373eaf',
 'total_steps': 500,
 'current_step': 34,
 'progress_timestamp': '2025-11-26T09:54:44.229000'}

### 列出所有訓練任務

In [6]:
url = f"{base_url}/training/list"
response = requests.get(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")

[{'job_id': '03b21a1729434c',
  'gpu_id': [0],
  'status': 'running',
  'config': {'task_type': 'instruction',
   'vram_budget_gb': 15.0,
   'select_multiple_gpus': False,
   'trainer_config': {'model_name_or_path': '/app/tmp/models/Qwen_Qwen2.5-0.5B-Instruct',
    'use_bfloat16': True,
    'use_flash_attn': False,
    'weight_decay': 0.01,
    'warmup_ratio': None,
    'lora_config': {'bias': 'none',
     'lora_alpha': 16,
     'lora_dropout': 0.05,
     'r': 8,
     'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj']},
    'max_epochs': 10,
    'batch_size': 1,
    'learning_rate': 2e-05,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 100,
    'logging_steps': 5,
    'val_check_interval': 1.0,
    'checkpoint_dir': './checkpoints',
    'log_dir': './logs',
    'accelerator': 'auto',
    'devices': 1,
    'strategy': 'auto',
    'precision': '16-mixed',
    'experiment_name': 'Qwen2.5_TaiwanChat_finetune',
    'run_name': '03b21a1729434c',
    'gradient_checkpointing

### 取消訓練任務

In [5]:
url = f"{base_url}/training/cancel/{job_id}"
response = requests.post(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")


錯誤: 400 - {"detail":"Job '481f1f210e2c45' is already in status: failed. Cannot cancel."}


### 刪除任務紀錄

In [109]:
url = f"{base_url}/training/delete/{job_id}"
response = requests.delete(url)

# 檢查回應狀態
if response.status_code == 200:
    result = response.json()
    display(result)
else:
    print(f"錯誤: {response.status_code} - {response.text}")

{'message': "Job ID '2b58f2021c874b' successfully deleted."}